# Comprehensive Exploratory Data Analysis of Human Trafficking Reports to the Disque 100 Hotline

## Step 1: Understand the Dataset

This first notebook documents the dataset before cleaning or analysis. Its priority is to establish the **unit of analysis**, validate the 62-column schema, classify variables into analytical domains, and create an auditable baseline for Step 2.

### Objectives

1. Load CSV, Excel, or Parquet safely.
2. Confirm dimensions, data types, memory usage, and coverage period.
3. Compare the observed schema with the expected schema.
4. Determine whether one row represents a complaint, violation, victim, suspect, or relationship.
5. Profile cardinality and inspect original categories without changing them.
6. Export structural documentation for the data-quality audit.

> Missing-value tokens, inconsistent accents, repeated spaces, invalid domains, and business-rule violations are preserved here and audited in Step 2.


## 1.1 Analytical unit

The working hypothesis is that `hash` identifies a complaint-related record. This must be tested. If a `hash` repeats while `violacao`, victim, suspect, or relationship fields change, the file may be at a lower grain than one complaint. Consequently, complaint totals should not be computed from row counts until the grain is confirmed.


In [5]:
from pathlib import Path
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)
print("Pandas:", pd.__version__)

Pandas: 3.1.1


## 1.2 Configure and load the source

Change `DATA_PATH` only. The original source is never overwritten.


In [6]:
from urllib.parse import urlparse

# DATA_PATH = "https://raw.githubusercontent.com/janaelsondias/eda-disque100/refs/heads/main/data/trafico-pessoas-com-outras-violacoes-2021-2025.csv"
DATA_PATH = Path("../data/trafico-pessoas-com-outras-violacoes-2021-2025.csv")
SHEET_NAME = 0
CSV_ENCODING = "utf-8"
CSV_SEPARATOR = ";"

def load_dataset(path, sheet_name=0, encoding="utf-8", separator=None):
    source = str(path)
    parsed = urlparse(source)
    is_url = parsed.scheme.lower() in {"http", "https"}
    ext = Path(parsed.path if is_url else source).suffix.lower()

    if ext == ".csv":
        options = {"sep": separator, "engine": "python" if separator is None else "c"}
        if separator is not None:
            options["low_memory"] = False
        try:
            return pd.read_csv(source, encoding=encoding, **options)
        except UnicodeDecodeError:
            return pd.read_csv(source, encoding="latin-1", **options)

    if ext in {".xlsx", ".xls"}:
        return pd.read_excel(source, sheet_name=sheet_name)
    if ext == ".parquet":
        return pd.read_parquet(source)
    raise ValueError("Supported formats: CSV, XLSX/XLS, and Parquet")

is_url = urlparse(str(DATA_PATH)).scheme.lower() in {"http", "https"}
if is_url or Path(DATA_PATH).exists():
    df_raw = load_dataset(DATA_PATH, SHEET_NAME, CSV_ENCODING, CSV_SEPARATOR)
    df = df_raw.copy(deep=True)
    print(f"Loaded {len(df):,} rows and {df.shape[1]:,} columns")
else:
    df_raw = df = None
    print(f"Update DATA_PATH. File not found: {Path(DATA_PATH).resolve()}")

Loaded 8,401 rows and 62 columns


## 1.3 Dimensions and memory

The supplied schema reports approximately 8,401 rows, 62 columns, and 8.2 MB. This cell verifies those values against the actual file.


In [7]:
if df is not None:
    overview = pd.DataFrame({
        "metric": ["Rows", "Columns", "Cells", "Memory (MB)"],
        "value": [len(df), df.shape[1], df.size, round(df.memory_usage(deep=True).sum()/1024**2, 2)]
    })
    display(overview)


,metric,value
0,Rows,8401.00
1,Columns,62.00
2,Cells,520862.00
3,Memory (MB),8.33


## 1.4 Safe observation preview

Inspect the beginning, end, and a random sample. `hash` is excluded from display to reduce unnecessary exposure of record identifiers.


In [8]:
if df is not None:
    preview_cols = [c for c in df.columns if c != "hash"]
    display(df[preview_cols].head(3))
    display(df[preview_cols].tail(3))
    display(df[preview_cols].sample(min(5, len(df)), random_state=42))


,data_de_cadastro,canal_de_atendimento,denuncia_emergencial,denunciante,cenario_da_violacao,pais,uf,municipio,frequencia,inicio_das_violacoes,sl_quantidade_vitimas,grupo_vulneravel,motivacao,relacao_vitima_suspeito,sl_vitima_cadastro,genero_da_vitima,orientacao_sexual_da_vitima,faixa_etaria_da_vitima,nacionalidade_da_vitima,sl_vitima_naturalidade,sl_vitima_naturalizado_uf,sl_vitima_naturalizado_municipio,deficiencia_da_vitima,doenca_rara_da_vitima,deficiencia_relacionada_a_doenca_rara,vitima_preso_a,pais_da_vitima,uf_da_vitima,municipio_da_vitima,profissao_da_vitima,grau_de_instrucao_da_vitima,religiao_da_vitima,raca_cor_da_vitima,etnia_da_vitima,faixa_de_renda_da_vitima,natureza_juridica_do_suspeito,genero_do_suspeito,orientacao_sexual_do_suspeito,faixa_etaria_do_suspeito,suspeito_nacionalidade,sl_suspeito_naturalidade,sl_suspeito_naturalizado_uf,sl_suspeito_naturalizado_municipio,deficiencia_do_suspeito,doenca_rara_do_suspeito,deficiencia_relacionada_a_doenca_rara_suspeito,suspeito_preso,pais_do_suspeito,uf_do_suspeito,sl_suspeito_municipio,profissao_do_suspeito,grau_de_instrucao_do_suspeito,religiao_do_suspeito,raca_cor_do_suspeito,suspeito_etnia,faixa_de_renda_do_suspeito,vinculo_orgao_pj_do_suspeito,sl_suspeito_ramo,suspeito_especificacao_etnia,vitima_especificacao_etnia,violacao
0,2021-01-08 23:11:00,TELEFÔNICO,NÃO,ANÔNIMO,SERVIÇO DE ABRIGAMENTO,BRASIL,SP,SÃO BERNARDO DO CAMPO,DIARIAMENTE,HÁ MAIS DE CINCO ANOS,10,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,PARA OBTENÇÃO DE BENEFÍCIO FINANCEIRO/ GANÂNCIA,OUTROS,PESSOA FÍSICA,FEMININO,N/D,NaN,N/D,NaN,NaN,NaN,N/D,NÃO TEM DOENÇA RARA,NaN,N/D,BRASIL,SP,SÃO BERNARDO DO CAMPO,N/D,N/D,N/D,N/D,N/D,N/D,PESSOA FÍSICA,MASCULINO,N/D,60 A 64 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO,NÃO,BRASIL,SP,SÃO BERNARDO DO CAMPO,N/D,N/D,N/D,N/D,N/D,N/D,NaN,N/D,N/D,N/D,INTEGRIDADE>PSÍQUICA>CALÚNIA
1,2021-01-14 07:11:00,TELEFÔNICO,NÃO,ANÔNIMO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,BRASIL,SP,SÃO PAULO,DIARIAMENTE,HÁ MAIS DE SEIS MESES,10,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,DA COABITAÇÃO/ CONVIVÊNCIA FAMILIAR/ RELAÇÃO AFETIVA,FAMILIARES,PESSOA FÍSICA,FEMININO,N/D,02 A 04 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,BRANCA,N/D,ATÉ 1 SM,PESSOA FÍSICA,FEMININO,N/D,35 A 39 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO,NÃO,BRASIL,SP,SÃO PAULO,N/D,ENSINO FUNDAMENTAL INCOMPLETO,N/D,BRANCA,N/D,ATÉ 1 SM,NaN,N/D,N/D,N/D,INTEGRIDADE>PSÍQUICA>AMEAÇA ou COAÇÃO
2,2021-01-12 07:34:00,TELEFÔNICO,NÃO,ANÔNIMO,CASA DA VÍTIMA,BRASIL,SP,SÃO PAULO,DIARIAMENTE,HÁ MAIS DE SEIS MESES,20,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,DA COABITAÇÃO/ CONVIVÊNCIA FAMILIAR/ RELAÇÃO AFETIVA,PAI,PESSOA FÍSICA,FEMININO,N/D,02 A 04 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NaN,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,BRANCA,N/D,N/D,PESSOA FÍSICA,MASCULINO,N/D,45 A 49 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO SABE INFORMAR,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,PRETA,N/D,N/D,NaN,N/D,N/D,N/D,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESSOAS - INTERNACIONAL


,data_de_cadastro,canal_de_atendimento,denuncia_emergencial,denunciante,cenario_da_violacao,pais,uf,municipio,frequencia,inicio_das_violacoes,sl_quantidade_vitimas,grupo_vulneravel,motivacao,relacao_vitima_suspeito,sl_vitima_cadastro,genero_da_vitima,orientacao_sexual_da_vitima,faixa_etaria_da_vitima,nacionalidade_da_vitima,sl_vitima_naturalidade,sl_vitima_naturalizado_uf,sl_vitima_naturalizado_municipio,deficiencia_da_vitima,doenca_rara_da_vitima,deficiencia_relacionada_a_doenca_rara,vitima_preso_a,pais_da_vitima,uf_da_vitima,municipio_da_vitima,profissao_da_vitima,grau_de_instrucao_da_vitima,religiao_da_vitima,raca_cor_da_vitima,etnia_da_vitima,faixa_de_renda_da_vitima,natureza_juridica_do_suspeito,genero_do_suspeito,orientacao_sexual_do_suspeito,faixa_etaria_do_suspeito,suspeito_nacionalidade,sl_suspeito_naturalidade,sl_suspeito_naturalizado_uf,sl_suspeito_naturalizado_municipio,deficiencia_do_suspeito,doenca_rara_do_suspeito,deficiencia_relacionada_a_doenca_rara_suspeito,suspeito_preso,pais_do_suspeito,uf_do_suspeito,sl_suspeito_municipio,profissao_do_suspeito,grau_de_instrucao_do_suspeito,religiao_do_suspeito,raca_cor_do_suspeito,suspeito_etnia,faixa_de_renda_do_suspeito,vinculo_orgao_pj_do_suspeito,sl_suspeito_ramo,suspeito_especificacao_etnia,vitima_especificacao_etnia,violacao
8398,2025-12-28 14:53:31,TELEFÔNICO,NÃO,TERCEIRO,LOCAL DE TRABALHO DA VÍTIMA,BR | BRASIL,SP,3547304 | SANTANA DE PARNAÍBA,DIARIAMENTE,HÁ MAIS DE DEZ ANOS,1,VIOLÊNCIA CONTRA A MULHER,OUTRO(S) ELEMENTO(S),OUTROS,PESSOA FÍSICA,FEMININO,NÃO,18 A 19 ANOS,BR | BRASIL,NaN,NaN,NaN,NÃO,NÃO,NaN,NÃO,BR | BRASIL,SP,3547304 | SANTANA DE PARNAÍBA,NaN,NaN,NaN,NÃO INFORMADO,NaN,NaN,PESSOA FÍSICA,FEMININO,NÃO,40 A 44 ANOS,BR | BRASIL,NaN,NaN,NaN,NÃO,NÃO,NaN,NÃO,BR | BRASIL,SP,3547304 | SANTANA DE PARNAÍBA,OUTROS NÃO LISTADOS,NaN,NaN,PRETA,NaN,NaN,NaN,NaN,NaN,NaN,INTEGRIDADE>FÍSICA>EXPOSIÇÃO DE RISCO À SAÚDE
8399,2025-12-30 15:31:58,WHATSAPP,NÃO,TERCEIRO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,BR | BRASIL,RS,4306106 | CRUZ ALTA,DIARIAMENTE,HÁ UMA SEMANA,1,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,OUTRO(S) ELEMENTO(S),CUIDADOR(A),PESSOA FÍSICA,FEMININO,NaN,MENOS DE 01 ANO,BR | BRASIL,NaN,NaN,NaN,NÃO,NÃO,NaN,NaN,BR | BRASIL,RS,4306106 | CRUZ ALTA,NaN,NaN,NaN,BRANCA,NaN,NaN,PESSOA FÍSICA,FEMININO,NÃO,45 A 49 ANOS,BR | BRASIL,NaN,NaN,NaN,NÃO,NÃO,NaN,NÃO,BR | BRASIL,RS,4306106 | CRUZ ALTA,CUIDADOR/BABÁ,NaN,NaN,NÃO INFORMADO,NaN,NaN,NaN,NaN,NaN,NaN,INTEGRIDADE>NEGLIGÊNCIA
8400,2025-12-30 15:31:58,WHATSAPP,NÃO,TERCEIRO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,BR | BRASIL,RS,4306106 | CRUZ ALTA,DIARIAMENTE,HÁ UMA SEMANA,1,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,NaN,CUIDADOR(A),PESSOA FÍSICA,FEMININO,NaN,MENOS DE 01 ANO,BR | BRASIL,NaN,NaN,NaN,NaN,NÃO,NaN,NaN,BR | BRASIL,RS,4306106 | CRUZ ALTA,NaN,NaN,NaN,BRANCA,NaN,NaN,PESSOA FÍSICA,FEMININO,NÃO,45 A 49 ANOS,BR | BRASIL,NaN,NaN,NaN,NaN,NÃO,NaN,NÃO,BR | BRASIL,RS,4306106 | CRUZ ALTA,CUIDADOR/BABÁ,NaN,NaN,NÃO INFORMADO,NaN,NaN,NaN,NaN,NaN,NaN,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE CRIANÇAS - NACIONAL


,data_de_cadastro,canal_de_atendimento,denuncia_emergencial,denunciante,cenario_da_violacao,pais,uf,municipio,frequencia,inicio_das_violacoes,sl_quantidade_vitimas,grupo_vulneravel,motivacao,relacao_vitima_suspeito,sl_vitima_cadastro,genero_da_vitima,orientacao_sexual_da_vitima,faixa_etaria_da_vitima,nacionalidade_da_vitima,sl_vitima_naturalidade,sl_vitima_naturalizado_uf,sl_vitima_naturalizado_municipio,deficiencia_da_vitima,doenca_rara_da_vitima,deficiencia_relacionada_a_doenca_rara,vitima_preso_a,pais_da_vitima,uf_da_vitima,municipio_da_vitima,profissao_da_vitima,grau_de_instrucao_da_vitima,religiao_da_vitima,raca_cor_da_vitima,etnia_da_vitima,faixa_de_renda_da_vitima,natureza_juridica_do_suspeito,genero_do_suspeito,orientacao_sexual_do_suspeito,faixa_etaria_do_suspeito,suspeito_nacionalidade,sl_suspeito_naturalidade,sl_suspeito_naturalizado_uf,sl_suspeito_naturalizado_municipio,deficiencia_do_suspeito,doenca_rara_do_suspeito,deficiencia_relacionada_a_doenca_rara_suspeito,suspeito_preso,pais_do_suspeito,uf_do_suspeito,sl_suspeito_municipio,profissao_do_suspeito,grau_de_instrucao_do_suspeito,religiao_do_suspeito,raca_cor_do_suspeito,suspeito_etnia,faixa_de_renda_do_suspeito,vinculo_orgao_pj_do_suspeito,sl_suspeito_ramo,suspeito_especificacao_etnia,vitima_especificacao_etnia,violacao
33,2021-01-08 07:02:00,TELEFÔNICO,NÃO,ANÔNIMO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,BRASIL,SP,SÃO PAULO,SEMANALMENTE,HÁ MAIS DE SEIS MESES,10,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,DA COABITAÇÃO/ CONVIVÊNCIA FAMILIAR/ RELAÇÃO AFETIVA,MÃE,PESSOA FÍSICA,FEMININO,N/D,02 A 04 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NaN,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,BRANCA,N/D,N/D,PESSOA FÍSICA,FEMININO,N/D,30 A 34 ANOS,N/D,NaN,NaN,NaN,N/D,NÃO TEM DOENÇA RARA,NÃO SABE INFORMAR,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,BRANCA,N/D,N/D,NaN,N/D,N/D,N/D,NaN
5091,2024-07-15 13:28:03,WHATSAPP,NÃO,TERCEIRO,OUTROS,BR | BRASIL,CE,2301604 | ASSARÉ,NÃO SABE INFORMAR,HÁ MAIS DE DEZ ANOS,1,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,NaN,OUTROS,PESSOA FÍSICA,FEMININO,NÃO,14 ANOS,BR | BRASIL,NaN,NaN,NaN,NaN,NÃO,NaN,NÃO,BR | BRASIL,DENUNCIANTE NÃO SOUBE INFORMAR,DENUNCIANTE NÃO SOUBE INFORMAR,NaN,NaN,NÃO SABE,AMARELA,NaN,NaN,ÓRGÃO PÚBLICO,NaN,NaN,NaN,BR | BRASIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BR | BRASIL,CE,2301604 | ASSARÉ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PODER EXECUTIVO EM GERAL - MUNICIPAL,NaN,NaN,INTEGRIDADE>PSÍQUICA>CONSTRANGIMENTO
5836,2024-11-12 19:34:45,TELEFÔNICO,NÃO,TERCEIRO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,BR | BRASIL,AL,2701704 | CAPELA,DIARIAMENTE,HÁ MAIS DE UM ANO,1,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,NaN,OUTROS,PESSOA FÍSICA,FEMININO,NÃO,15 ANOS,BR | BRASIL,NaN,NaN,NaN,NaN,NÃO,NaN,NÃO,BR | BRASIL,AL,2701704 | CAPELA,NaN,NaN,NaN,BRANCA,NaN,NaN,PESSOA FÍSICA,FEMININO,NÃO,50 A 54 ANOS,BR | BRASIL,NaN,NaN,NaN,NaN,NÃO,NaN,NÃO,BR | BRASIL,AL,2701704 | CAPELA,ADMINISTRADOR,NaN,NaN,BRANCA,NaN,NaN,NaN,NaN,NaN,NaN,INTEGRIDADE>PSÍQUICA>EXPOSIÇÃO (EROTIZAÇÃO)
940,2023-02-16 19:58:33,TELEFÔNICO,RISCO IMINENTE DE MORTE DA VÍTIMA,TERCEIRO,CASA DE FAMILIARES,BR | BRASIL,SP,351550 | FERNANDÓPOLIS,DIARIAMENTE,HÁ UM MÊS,1,"VIOLÊNCIA CONTRA CIDADÃO, FAMÍLIA OU COMUNIDADE",NaN,OUTROS FAMILIARES,PESSOA FÍSICA,MASCULINO,NÃO,NÃO INFORMADO,BR | BRASIL,NaN,NaN,NaN,NaN,NÃO,NaN,NÃO,BR | BRASIL,SP,351550 | FERNANDÓPOLIS,OUTROS NÃO LISTADOS,ENSINO FUNDAMENTAL INCOMPLETO,NaN,NaN,NaN,NaN,COMUNIDADE/FAMÍLIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BR | BRASIL,SP,351550 | FERNANDÓPOLIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESSOAS - INTERNACIONAL
1870,2024-03-14 23:25:10,TELEFÔNICO,NÃO,TERCEIRO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,BR | BRASIL,RN,2412005 | SÃO GONÇALO DO AMARANTE,DIARIAMENTE,HÁ MAIS DE UM ANO,1,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,MOTIVAÇÃO.PARA FINS DE REMOÇÃO DE ÓRGÃOS/TRÁFICO DE ÓRGÃOS,OUTROS,COMUNIDADE/FAMÍLIA,NaN,NÃO,13 ANOS,BR | BRASIL,NaN,NaN,NaN,NÃO,NaN,NaN,NÃO,BR | BRASIL,RN,2412005 | SÃO GONÇALO DO AMARANTE,NaN,N

## 1.5 Structural profile

This is a baseline based on physical nulls only. Semantic missing values such as `NÃO INFORMADO`, `SEM INFORMAÇÃO`, `-99`, and `-999` remain untouched for Step 2.


In [9]:
def sample_values(s, n=5):
    return " | ".join(s.dropna().astype(str).unique()[:n])

if df is not None:
    structural_profile = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [int(df[c].notna().sum()) for c in df.columns],
        "physical_null": [int(df[c].isna().sum()) for c in df.columns],
        "physical_null_pct": [round(df[c].isna().mean()*100, 2) for c in df.columns],
        "unique_non_null": [int(df[c].nunique(dropna=True)) for c in df.columns],
        "sample_values": [sample_values(df[c]) for c in df.columns]
    })
    structural_profile["unique_pct_rows"] = (structural_profile["unique_non_null"]/len(df)*100).round(2)
    display(structural_profile)


,column,dtype,non_null,physical_null,physical_null_pct,unique_non_null,sample_values,unique_pct_rows
0,hash,str,8401,0,0.00,1263,79c48e0d422d3ab1a306f8d7e498b8be | 0b9e67b28384351a4006c49ba3ff66b7 | 6fa1ed73ec4c3f385683c9fc03a3c062 | ba12fedf0ff...,15.03
1,data_de_cadastro,str,8401,0,0.00,640,2021-01-08 23:11:00 | 2021-01-14 07:11:00 | 2021-01-12 07:34:00 | 2021-01-13 02:28:00 | 2021-01-13 18:24:00,7.62
2,canal_de_atendimento,str,8401,0,0.00,8,TELEFÔNICO | CARTA | WHATSAPP | TELEGRAM | WEBCHAT,0.10
3,denuncia_emergencial,str,8401,0,0.00,5,NÃO | SITUAÇÃO FLAGRANTE - ATÉ 24H DA OCORRÊNCIA - SEM FREQUÊNCIA | N/D | RISCO IMINENTE DE MORTE DA VÍTIMA | VÍTIMA...,0.06
4,denunciante,str,8401,0,0.00,6,ANÔNIMO | TERCEIRO | A PRÓPRIA VÍTIMA | PESSOA COM DEFICIÊNCIA MENTAL | PESSOA EM SOFRIMENTO PSÍQUICO,0.07
5,cenario_da_violacao,str,8401,0,0.00,28,SERVIÇO DE ABRIGAMENTO | CASA ONDE RESIDE A VÍTIMA E O SUSPEITO | CASA DA VÍTIMA | VIA PÚBLICA | OUTROS,0.33
6,pais,str,8401,0,0.00,29,BRASIL | PANAMÁ | ESTADOS UNIDOS | ALEMANHA | BR | BRASIL,0.35
7,uf,str,7828,573,6.82,30,SP | RJ | SE | MG | PB,0.36
8,municipio,str,7810,591,7.03,313,SÃO BERNARDO DO CAMPO | SÃO PAULO | RIO DE JANEIRO | SÃO DOMINGOS | IBIAÍ,3.73
9,frequencia,str,8401,0,0.00,7,DIARIAMENTE | ÚNICA OCORRÊNCIA | SEMANALMENTE | OCASIONALMENTE | N/D,0.08


## 1.6 Validate the expected 62-column schema


In [10]:
EXPECTED_COLUMNS = [
"hash","data_de_cadastro","canal_de_atendimento","denuncia_emergencial","denunciante","cenario_da_violacao","pais","uf","municipio","frequencia","inicio_das_violacoes","sl_quantidade_vitimas","grupo_vulneravel","motivacao","relacao_vitima_suspeito","sl_vitima_cadastro","genero_da_vitima","orientacao_sexual_da_vitima","faixa_etaria_da_vitima","nacionalidade_da_vitima","sl_vitima_naturalidade","sl_vitima_naturalizado_uf","sl_vitima_naturalizado_municipio","deficiencia_da_vitima","doenca_rara_da_vitima","deficiencia_relacionada_a_doenca_rara","vitima_preso_a","pais_da_vitima","uf_da_vitima","municipio_da_vitima","profissao_da_vitima","grau_de_instrucao_da_vitima","religiao_da_vitima","raca_cor_da_vitima","etnia_da_vitima","faixa_de_renda_da_vitima","natureza_juridica_do_suspeito","genero_do_suspeito","orientacao_sexual_do_suspeito","faixa_etaria_do_suspeito","suspeito_nacionalidade","sl_suspeito_naturalidade","sl_suspeito_naturalizado_uf","sl_suspeito_naturalizado_municipio","deficiencia_do_suspeito","doenca_rara_do_suspeito","deficiencia_relacionada_a_doenca_rara_suspeito","suspeito_preso","pais_do_suspeito","uf_do_suspeito","sl_suspeito_municipio","profissao_do_suspeito","grau_de_instrucao_do_suspeito","religiao_do_suspeito","raca_cor_do_suspeito","suspeito_etnia","faixa_de_renda_do_suspeito","vinculo_orgao_pj_do_suspeito","sl_suspeito_ramo","suspeito_especificacao_etnia","vitima_especificacao_etnia","violacao"]

def normalize_name(x):
    x = unicodedata.normalize("NFKD", str(x))
    x = "".join(ch for ch in x if not unicodedata.combining(ch))
    return re.sub(r"[^0-9a-zA-Z]+", "_", x.strip().lower()).strip("_")

if df is not None:
    observed = {normalize_name(c) for c in df.columns}
    expected = set(EXPECTED_COLUMNS)
    schema_comparison = pd.DataFrame({
        "status": ["missing_expected"]*len(expected-observed) + ["unexpected"]*len(observed-expected),
        "column": sorted(expected-observed) + sorted(observed-expected)
    })
    display(Markdown("**All 62 expected columns are present.**") if schema_comparison.empty else schema_comparison)


**All 62 expected columns are present.**

## 1.7 Variable domains

Variables are grouped into complaint, location, victim, suspect, relationship, and identifier domains. This separation is used throughout the rest of the EDA.


In [11]:
DOMAIN_MAP = {
"identifier": ["hash"],
"complaint": ["data_de_cadastro","canal_de_atendimento","denuncia_emergencial","denunciante","cenario_da_violacao","frequencia","inicio_das_violacoes","sl_quantidade_vitimas","grupo_vulneravel","motivacao","violacao"],
"complaint_location": ["pais","uf","municipio"],
"relationship": ["relacao_vitima_suspeito"],
"victim": [c for c in EXPECTED_COLUMNS if "vitima" in c and c != "relacao_vitima_suspeito"],
"suspect": [c for c in EXPECTED_COLUMNS if "suspeito" in c and c != "relacao_vitima_suspeito"]
}
if df is not None:
    rows=[]
    observed={normalize_name(c) for c in df.columns}
    for domain, cols in DOMAIN_MAP.items():
        for c in cols: rows.append({"domain":domain,"column":c,"present":c in observed})
    variable_domains=pd.DataFrame(rows)
    display(variable_domains.groupby("domain").agg(expected=("column","size"), present=("present","sum")).reset_index())


,domain,expected,present
0,complaint,11,11
1,complaint_location,3,3
2,identifier,1,1
3,relationship,1,1
4,suspect,24,24
5,victim,22,22


## 1.8 Preliminary analytical roles


In [13]:
if df is not None:
    roles=[]
    for c in df.columns:
        u=df[c].nunique(dropna=True)
        if normalize_name(c)=="hash" or u >= .98*len(df): role="identifier_candidate"
        elif pd.api.types.is_datetime64_any_dtype(df[c]): role="datetime"
        elif pd.api.types.is_numeric_dtype(df[c]): role="numeric"
        elif u <= 2: role="binary_categorical_candidate"
        elif u <= 50: role="categorical_candidate"
        else: role="high_cardinality_text_or_category"
        roles.append({"column":c,"unique_non_null":u,"preliminary_role":role})
    analytical_roles=pd.DataFrame(roles)
    display(analytical_roles)


,column,unique_non_null,preliminary_role
0,hash,1263,identifier_candidate
1,data_de_cadastro,640,high_cardinality_text_or_category
2,canal_de_atendimento,8,categorical_candidate
3,denuncia_emergencial,5,categorical_candidate
4,denunciante,6,categorical_candidate
5,cenario_da_violacao,28,categorical_candidate
6,pais,29,categorical_candidate
7,uf,30,categorical_candidate
8,municipio,313,high_cardinality_text_or_category
9,frequencia,7,categorical_candidate


## 1.9 Test the unit-of-analysis hypothesis

This is the most important Step 1 test. If `hash` repeats, inspect which attributes vary within each repeated identifier.


In [29]:
# df.duplicated(subset=)
cols_without_hash_violation = df.columns.difference(["hash", "violacao"])
cols_without_hash = df.columns.difference(["hash"])
display(df.duplicated(subset=cols_without_hash).sum())
display(df.duplicated(subset=cols_without_hash_violation).sum())
display(df.duplicated().sum())
display(df.duplicated(subset=["hash"]).sum())
display(df.duplicated(subset=["hash", "data_de_cadastro"]).sum())

np.int64(930)

np.int64(4909)

np.int64(0)

np.int64(7138)

np.int64(7138)

In [ ]:
# how many violations are there per report
df.groupby("hash")["violacao"].nunique().describe()

count    1263.000000
mean        6.004751
std         4.787892
min         1.000000
25%         3.000000
50%         5.000000
75%         8.000000
max        48.000000
Name: violacao, dtype: float64

In [33]:
# how many total lines there are per report.
df.groupby("hash").size().describe()

count    1263.000000
mean        6.651623
std         6.762833
min         1.000000
25%         3.000000
50%         5.000000
75%         8.000000
max       140.000000
dtype: float64

In [21]:
if df is not None and "hash" in df.columns:
    hash_summary=pd.DataFrame({"metric":["Rows","Non-null hashes","Unique hashes","Rows belonging to repeated hashes"],"value":[len(df),df["hash"].notna().sum(),df["hash"].nunique(dropna=True),(df["hash"].notna() & df["hash"].duplicated(keep=False)).sum()]})
    display(hash_summary)
    repeated=df[df["hash"].notna() & df["hash"].duplicated(keep=False)]
    if not repeated.empty:
        inspect=[c for c in ["hash","violacao","sl_quantidade_vitimas","relacao_vitima_suspeito","genero_da_vitima","genero_do_suspeito"] if c in df]
        display(repeated[inspect].sort_values("hash").head(30))


,metric,value
0,Rows,8401
1,Non-null hashes,8401
2,Unique hashes,1263
3,Rows belonging to repeated hashes,8304


,hash,violacao,sl_quantidade_vitimas,relacao_vitima_suspeito,genero_da_vitima,genero_do_suspeito
5301,002F819E35D44A499A7A94986A57219BF5D730B561452B41545964AF67F343D8,INTEGRIDADE>PSÍQUICA>TORTURA PSÍQUICA,1,OUTROS,NÃO INFORMADO,NaN
5304,002F819E35D44A499A7A94986A57219BF5D730B561452B41545964AF67F343D8,INTEGRIDADE>PSÍQUICA>AMEAÇA ou COAÇÃO,1,OUTROS,NÃO INFORMADO,NaN
5305,002F819E35D44A499A7A94986A57219BF5D730B561452B41545964AF67F343D8,INTEGRIDADE>PSÍQUICA>CONSTRANGIMENTO,1,OUTROS,NÃO INFORMADO,NaN
5307,002F819E35D44A499A7A94986A57219BF5D730B561452B41545964AF67F343D8,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESSOAS - NACIONAL,1,OUTROS,NÃO INFORMADO,NaN
5309,002F819E35D44A499A7A94986A57219BF5D730B561452B41545964AF67F343D8,INTEGRIDADE>FÍSICA>OUTROS,1,OUTROS,NÃO INFORMADO,NaN
697,005929E4587AF2DA96A76E5590A542467A1E397D0FAAEEDE13ABF1ADC110883B,INTEGRIDADE>PSÍQUICA>EXPOSIÇÃO (EROTIZAÇÃO),1,NÃO SABE INFORMAR,FEMININO,FEMININO
698,005929E4587AF2DA96A76E5590A542467A1E397D0FAAEEDE13ABF1ADC110883B,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESSOAS - NACIONAL,1,NÃO SABE INFORMAR,FEMININO,FEMININO
699,005929E4587AF2DA96A76E5590A542467A1E397D0FAAEEDE13ABF1ADC110883B,LIBERDADE>SEXUAL>FÍSICA>EXPLORAÇÃO SEXUAL,1,NÃO SABE INFORMAR,FEMININO,FEMININO
700,005929E4587AF2DA96A76E5590A542467A1E397D0FAAEEDE13ABF1ADC110883B,LIBERDADE>SEXUAL>PSÍQUICA>ABUSO SEXUAL PSÍQUICO,1,NÃO SABE INFORMAR,FEMININO,FEMININO
701,005929E4587AF2DA96A76E5590A542467A1E397D0FAAEEDE13ABF1ADC110883B,LIBERDADE>SEXUAL>PSÍQUICA>ASSÉDIO SEXUAL,1,NÃO SABE INFORMAR,FEMININO,FEMININO


The hash identifies a report and each row represent an human rights violation.

Multiplicidade legítima de registros decorrente da modelagem relacional do Disque 100. O identificador hash representa a denúncia e pode estar associado a múltiplas violações, motivações, agravantes e combinações de perfis de vítima e suspeito. Consequentemente, a repetição de valores em diversas colunas não caracteriza duplicidade de dados, mas sim relacionamentos um-para-muitos previstos pela documentação do sistema.


In [65]:
violation_levels = df["violacao"].str.split(">", expand=True)

taxonomy = ["classe", "subclasse", "especie", "subespecie"]
extra_levels = max(0, violation_levels.shape[1] - len(taxonomy))
taxonomy += [f"subespecie_{n}" for n in range(1, extra_levels + 1)]

violation_levels = violation_levels.reindex(columns=range(len(taxonomy)))
violation_levels.columns = taxonomy

violation_levels["subespecie"] = (
    violation_levels["subespecie"]
    .fillna(violation_levels["subclasse"])
    .fillna(violation_levels["classe"] if violation_levels["classe"].notna().any() else pd.NA)
)

if violation_levels["subespecie"].isna().any():
    raise ValueError("Há registros sem subespécie, subclasse e classe.")

display(violation_levels.head())
display(violation_levels.isna().sum())

ValueError: Há registros sem subespécie, subclasse e classe.

In [59]:
df.isna().mean()

hash                                              0.000000
data_de_cadastro                                  0.000000
canal_de_atendimento                              0.000000
denuncia_emergencial                              0.000000
denunciante                                       0.000000
cenario_da_violacao                               0.000000
pais                                              0.000000
uf                                                0.068206
municipio                                         0.070349
frequencia                                        0.000000
inicio_das_violacoes                              0.000000
sl_quantidade_vitimas                             0.000000
grupo_vulneravel                                  0.000000
motivacao                                         0.650518
relacao_vitima_suspeito                           0.000000
sl_vitima_cadastro                                0.000000
genero_da_vitima                                  0.1349

## 1.10 Inspect original categorical values

Original values are displayed without normalization so that suspicious differences remain visible.


In [35]:
def frequency_table(data, column, top_n=20):
    t=data[column].value_counts(dropna=False).head(top_n).rename("count").to_frame()
    t["percent_rows"]=(t["count"]/len(data)*100).round(2)
    return t.reset_index(names=column)

if df is not None:
    priority=["canal_de_atendimento","denuncia_emergencial","cenario_da_violacao","frequencia","grupo_vulneravel","relacao_vitima_suspeito","genero_da_vitima","faixa_etaria_da_vitima","raca_cor_da_vitima","genero_do_suspeito","faixa_etaria_do_suspeito","violacao","uf"]
    for c in priority:
        if c in df:
            display(Markdown(f"### `{c}`")); display(frequency_table(df,c))


### `canal_de_atendimento`

,canal_de_atendimento,count,percent_rows
0,TELEFÔNICO,6091,72.50
1,WHATSAPP,1495,17.80
2,E-MAIL,641,7.63
3,PROCESSO SEI,84,1.00
4,WEBCHAT,46,0.55
5,PORTAL,29,0.35
6,TELEGRAM,14,0.17
7,CARTA,1,0.01


### `denuncia_emergencial`

,denuncia_emergencial,count,percent_rows
0,NÃO,7646,91.01
1,RISCO IMINENTE DE MORTE DA VÍTIMA,634,7.55
2,SITUAÇÃO FLAGRANTE - ATÉ 24H DA OCORRÊNCIA - SEM FREQUÊNCIA,115,1.37
3,VÍTIMA EM SANGRAMENTO,5,0.06
4,N/D,1,0.01


### `cenario_da_violacao`

,cenario_da_violacao,count,percent_rows
0,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,1776,21.14
1,OUTROS,1721,20.49
2,CASA DA VÍTIMA,1225,14.58
3,CASA DO SUSPEITO,1088,12.95
4,CASA DE FAMILIARES,434,5.17
5,ESTABELECIMENTO COMERCIAL,357,4.25
6,AMBIENTE VIRTUAL (NO ÂMBITO DA INTERNET),333,3.96
7,VIA PÚBLICA,290,3.45
8,NÃO SABE INFORMAR,242,2.88
9,LOCAL DE TRABALHO DA VÍTIMA,208,2.48


... (16 outputs left out)

### `violacao`

,violacao,count,percent_rows
0,INTEGRIDADE>PSÍQUICA>TORTURA PSÍQUICA,535,6.37
1,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE CRIANÇAS - NACIONAL,524,6.24
2,INTEGRIDADE>FÍSICA>EXPOSIÇÃO DE RISCO À SAÚDE,505,6.01
3,INTEGRIDADE>NEGLIGÊNCIA,496,5.90
4,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESSOAS - NACIONAL,381,4.54
5,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESSOAS - INTERNACIONAL,372,4.43
6,LIBERDADE>SEXUAL>FÍSICA>EXPLORAÇÃO SEXUAL,352,4.19
7,INTEGRIDADE>PSÍQUICA>AMEAÇA ou COAÇÃO,341,4.06
8,INTEGRIDADE>PSÍQUICA>CONSTRANGIMENTO,332,3.95
9,INTEGRIDADE>PSÍQUICA>EXPOSIÇÃO,295,3.51


### `uf`

,uf,count,percent_rows
0,SP,1797,21.39
1,MG,975,11.61
2,DENUNCIANTE NÃO SOUBE INFORMAR,773,9.20
3,RJ,611,7.27
4,NaN,573,6.82
5,RS,514,6.12
6,CE,325,3.87
7,BA,287,3.42
8,PR,276,3.29
9,PE,231,2.75


## 1.11 Date and quantitative fields


In [26]:
if df is not None:
    if "data_de_cadastro" in df:
        d=pd.to_datetime(df["data_de_cadastro"],errors="coerce")
        display(pd.DataFrame({"metric":["Original non-null","Parsed","Unparseable","Minimum","Maximum"],"value":[df["data_de_cadastro"].notna().sum(),d.notna().sum(),(df["data_de_cadastro"].notna()&d.isna()).sum(),d.min(),d.max()]}))
    if "sl_quantidade_vitimas" in df:
        q=pd.to_numeric(df["sl_quantidade_vitimas"],errors="coerce")
        display(q.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).to_frame())
    if "vinculo_orgao_pj_do_suspeito" in df:
        v=df["vinculo_orgao_pj_do_suspeito"].astype(str)
        display(f"Quantidade de valores de 'vinculo_orgao_pj_do_suspeito' {v.notna().sum().sum()}")


,metric,value
0,Original non-null,8401
1,Parsed,8401
2,Unparseable,0
3,Minimum,2021-01-05 08:28:00
4,Maximum,2025-12-30 15:31:58


,sl_quantidade_vitimas
count,8401.000000
mean,3.287704
std,7.467195
min,1.000000
1%,1.000000
5%,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
95%,10.000000


"Quantidade de valores de 'vinculo_orgao_pj_do_suspeito' 0"

## 1.12 Initial data dictionary and export

Descriptions inferred from names are provisional and must be validated against the official codebook or collection form.


In [27]:
DESCRIPTION={"hash":"Record or complaint identifier; uniqueness must be verified.","data_de_cadastro":"Registration date and time.","canal_de_atendimento":"Complaint intake channel.","denuncia_emergencial":"Emergency classification indicator.","sl_quantidade_vitimas":"Number of victims associated with the record.","grupo_vulneravel":"Vulnerable group associated with the complaint.","relacao_vitima_suspeito":"Reported victim-suspect relationship.","violacao":"Reported human rights violation category."}
if df is not None:
    lookup={v:k for k,vals in DOMAIN_MAP.items() for v in vals}
    data_dictionary=structural_profile.copy()
    data_dictionary["normalized_name"]=data_dictionary["column"].map(normalize_name)
    data_dictionary["domain"]=data_dictionary["normalized_name"].map(lookup).fillna("unclassified")
    data_dictionary["description"]=data_dictionary["normalized_name"].map(DESCRIPTION).fillna("Pending validation against official metadata.")
    display(data_dictionary[["column","normalized_name","domain","description","dtype","non_null","physical_null_pct","unique_non_null","sample_values"]])
    out=Path("outputs/step_01_understand_dataset"); out.mkdir(parents=True,exist_ok=True)
    structural_profile.to_csv(out/"structural_profile.csv",index=False,encoding="utf-8-sig")
    analytical_roles.to_csv(out/"analytical_roles.csv",index=False,encoding="utf-8-sig")
    data_dictionary.to_csv(out/"initial_data_dictionary.csv",index=False,encoding="utf-8-sig")
    schema_comparison.to_csv(out/"schema_comparison.csv",index=False,encoding="utf-8-sig")
    print("Saved to",out.resolve())


,column,normalized_name,domain,description,dtype,non_null,physical_null_pct,unique_non_null,sample_values
0,hash,hash,identifier,Record or complaint identifier; uniqueness must be verified.,str,8401,0.00,1263,79c48e0d422d3ab1a306f8d7e498b8be | 0b9e67b28384351a4006c49ba3ff66b7 | 6fa1ed73ec4c3f385683c9fc03a3c062 | ba12fedf0ff...
1,data_de_cadastro,data_de_cadastro,complaint,Registration date and time.,str,8401,0.00,640,2021-01-08 23:11:00 | 2021-01-14 07:11:00 | 2021-01-12 07:34:00 | 2021-01-13 02:28:00 | 2021-01-13 18:24:00
2,canal_de_atendimento,canal_de_atendimento,complaint,Complaint intake channel.,str,8401,0.00,8,TELEFÔNICO | CARTA | WHATSAPP | TELEGRAM | WEBCHAT
3,denuncia_emergencial,denuncia_emergencial,complaint,Emergency classification indicator.,str,8401,0.00,5,NÃO | SITUAÇÃO FLAGRANTE - ATÉ 24H DA OCORRÊNCIA - SEM FREQUÊNCIA | N/D | RISCO IMINENTE DE MORTE DA VÍTIMA | VÍTIMA...
4,denunciante,denunciante,complaint,Pending validation against official metadata.,str,8401,0.00,6,ANÔNIMO | TERCEIRO | A PRÓPRIA VÍTIMA | PESSOA COM DEFICIÊNCIA MENTAL | PESSOA EM SOFRIMENTO PSÍQUICO
5,cenario_da_violacao,cenario_da_violacao,complaint,Pending validation against official metadata.,str,8401,0.00,28,SERVIÇO DE ABRIGAMENTO | CASA ONDE RESIDE A VÍTIMA E O SUSPEITO | CASA DA VÍTIMA | VIA PÚBLICA | OUTROS
6,pais,pais,complaint_location,Pending validation against official metadata.,str,8401,0.00,29,BRASIL | PANAMÁ | ESTADOS UNIDOS | ALEMANHA | BR | BRASIL
7,uf,uf,complaint_location,Pending validation against official metadata.,str,7828,6.82,30,SP | RJ | SE | MG | PB
8,municipio,municipio,complaint_location,Pending validation against official metadata.,str,7810,7.03,313,SÃO BERNARDO DO CAMPO | SÃO PAULO | RIO DE JANEIRO | SÃO DOMINGOS | IBIAÍ
9,frequencia,frequencia,complaint,Pending validation against official metadata.,str,8401,0.00,7,DIARIAMENTE | ÚNICA OCORRÊNCIA | SEMANALMENTE | OCASIONALMENTE | N/D


Saved to C:\Projetos\eda-disque100-completo\eda\outputs\step_01_understand_dataset


## 1.13 Step 1 conclusions checklist

After execution, document:

- Whether one row is a complaint, violation, victim, suspect, or relationship.
- Whether `hash` is complete and unique.
- Whether totals should use row count or unique `hash`.
- The earliest and latest `data_de_cadastro`.
- Whether all 62 columns are present.
- Which variables are conditional and therefore legitimately sparse.
- Which categorical variables need official allowed-value domains.
- Whether any fields require additional privacy controls.

### Next step

**Step 2: Data Quality Assessment** will detect physical and semantic missingness, `-95/-97/-98/-99/-999`, capitalization and accent variants, double spaces, invalid domains, duplicate structures, date anomalies, UF and municipality inconsistencies, and cross-field business-rule violations.
